# CSS 551 · HW7 · A gradient step, a fit, and a digit's DCT (Python track)

Fill in every function marked `TODO`, run all cells, and submit this notebook **run**, with
the check cell's output visible. The functions and the numbers are the same as the
JavaScript track's (see the homework page); the last cell compares yours with them.
`gradients` is given: do not edit it. You write `forward`, `sgd_step`, `train`, `dct2`,
`idct2_truncated` and `psnr`, and the learning-rate report (10 points, read by the grader).

Rules: NumPy is allowed for arrays, dot products and matrix products. No `scipy`, no
autodiff (`torch`, `jax`), no `np.fft` or DCT routines: the point is to write the training
loop and the transform yourself. Keep `data.json` and `expected.json` next to this notebook.

In [ ]:
import json, math
import numpy as np

DATA = json.load(open("data.json"))
EXP = json.load(open("expected.json"))
INPUTS = EXP["inputs"]

In [ ]:
def forward(m, x):
    """m = {"w1", "b1", "w2" (arrays of H), "b2"}; returns (a, f) with
    a[h] = tanh(w1[h] x + b1[h]) and f = b2 + sum_h w2[h] a[h]."""
    # TODO
    return np.zeros(len(m["w1"])), 0.0

In [ ]:
# ---- GIVEN: do not edit gradients(). ----
def gradients(m, xs, ys):
    """Returns {"w1", "b1", "w2" (arrays), "b2", "loss"}: the MEAN squared error
    loss = (1 / n) sum_i (f(xs[i]) - ys[i])^2 and its gradient with respect to each parameter,
    by backpropagation. It runs its own forward pass, so it works before your forward() does."""
    w1, b1, w2 = (np.asarray(m[k], float) for k in ("w1", "b1", "w2"))
    g = {"w1": np.zeros_like(w1), "b1": np.zeros_like(b1), "w2": np.zeros_like(w2), "b2": 0.0, "loss": 0.0}
    n = len(xs)
    for x, y in zip(xs, ys):
        a = np.tanh(w1 * x + b1)
        err = float(m["b2"] + np.dot(w2, a)) - y
        g["loss"] += err * err / n
        dF = 2 * err / n
        g["b2"] += dF
        g["w2"] += dF * a
        dz = dF * w2 * (1 - a * a)
        g["w1"] += dz * x
        g["b1"] += dz
    return g
    # ---- end of the given code ----

In [ ]:
def sgd_step(m, g, lr):
    """A NEW model with every parameter moved by -lr times its gradient."""
    # TODO
    return {k: np.array(m[k], float) for k in ("w1", "b1", "w2")} | {"b2": m["b2"]}

In [ ]:
def train(m, xs, ys, lr, steps):
    """A list of steps + 1 losses: item 0 is the loss of m before any step, item k the loss after
    k full-batch steps. Each step is g = gradients(m, xs, ys), then m = sgd_step(m, g, lr).
    Do not change the model you were given: keep your own current model in a variable."""
    # TODO
    return []

In [ ]:
def dct2(img, n):
    """Orthonormal 2D DCT-II of an n x n image given row-major (img[y * n + x]).
    Returns C, row-major, C[u * n + v] for vertical frequency u, horizontal v:
    C[u][v] = c(u) c(v) sum_y sum_x img[y][x] cos(pi (x + 0.5) v / n) cos(pi (y + 0.5) u / n),
    c(0) = sqrt(1/n), c(k > 0) = sqrt(2/n). (Loops or a matrix product; not scipy.)"""
    # TODO
    return np.zeros(n * n)

In [ ]:
def idct2_truncated(C, n, k):
    """The image rebuilt from the low-frequency k x k block of C only (u < k and v < k), row-major."""
    # TODO
    return np.zeros(n * n)

In [ ]:
def psnr(a, b):
    """PSNR in dB of b against a, pixel values in [0, 1]: 10 log10(1 / MSE)."""
    # TODO
    return 0.0

## Learning-rate report (10 points, read by the grader)

Run the cell below (change the rates and the number of steps as you like), then write 3 to 5
sentences in the next cell: which rate was too slow, which diverged, and why, in
terms of the size of the step. The grader reads this cell; it is not checked by the code.

In [ ]:
xs, ys = DATA["samples"]["xs"], DATA["samples"]["ys"]
for lr in (0.002, 0.05, 0.4):
    start_model = {k: np.array(DATA["init"][k], float) for k in ("w1", "b1", "w2")} | {"b2": DATA["init"]["b2"]}
    h = train(start_model, xs, ys, lr, 40)
    if len(h) > 40:
        print(f"lr {lr:<6} loss at steps 0, 5, 10, 20, 40: " + ", ".join(f"{h[i]:.4g}" for i in (0, 5, 10, 20, 40)))

### Your report

(write 3 to 5 sentences here)

## Checks

Run this last. It prints each check with your value and the expected one, and the score on the published inputs. Grading also runs your functions on inputs that are not published.

In [ ]:
def flat(v):
    if isinstance(v, dict):
        raise TypeError("a check returned a dict; return the values in the order the check asks for")
    if isinstance(v, (list, tuple, np.ndarray)):
        out = []
        for x in v:
            out.extend(flat(x))
        return out
    if isinstance(v, (bool, np.bool_)):
        return [bool(v)]
    return [float(v)]

def same(got, want, tol):
    if want is None or got is None:
        return got is None and want is None
    g, w = flat(got), flat(want)
    if len(g) != len(w):
        return False
    return all(abs(a - b) <= tol and not math.isnan(a) for a, b in zip(g, w))

def report(runs):
    score, lines = 0, []
    for c in EXP["checks"]:
        try:
            got, err = runs[c["id"]](), None
        except Exception as e:  # a stub or a bug: the check fails, the rest still run
            got, err = None, repr(e)
        ok = err is None and same(got, c["expected"], c["tol"])
        score += c["points"] if ok else 0
        shown = err if err else np.round(np.array(flat(got), dtype=float), 4).tolist() if got is not None else None
        lines.append(f'{"PASS" if ok else "FAIL"}  {c["id"]:<8} {c["points"]:>3} pts  {c["label"]}\n      yours {shown}\n      expected {c["expected"]}')
    print("\n".join(lines))
    top = sum(c["points"] for c in EXP["checks"])
    for h in EXP.get("hand", []):
        print(f'BY HAND  {h["id"]:<8} {h["points"]:>3} pts  {h["label"]}: not run here, read by the grader')
    print(f"\nscore on the published inputs: {score} / {top}")
    if score == top:
        print("ALL CHECKS PASS")


T = INPUTS["tiny"]
digit, n = INPUTS.get("digit", DATA["digit"]), INPUTS["dct"]["n"]

def start():
    return {k: np.array(DATA["init"][k], float) for k in ("w1", "b1", "w2")} | {"b2": DATA["init"]["b2"]}

def fit_run():
    h = train(start(), DATA["samples"]["xs"], DATA["samples"]["ys"], INPUTS["fit"]["lr"], INPUTS["fit"]["steps"])
    return [len(h), h[0], h[-1]]

def lr_run():
    L = INPUTS["lr"]
    xs, ys = DATA["samples"]["xs"], DATA["samples"]["ys"]
    slow, good, big = (train(start(), xs, ys, lr, L["steps"]) for lr in L["rates"])
    # the first step whose loss is above blowup times the start (NaN counts as above); -1 if none
    first = next((k for k in range(1, len(big)) if not big[k] <= L["blowup"] * big[0]), -1)
    return [slow[-1], good[-1], first]

report({
    "fwd": lambda: list(forward(T, T["x"])),
    "after": lambda: forward(sgd_step(T, gradients(T, [T["x"]], [T["y"]]), T["lr"]), T["x"])[1],
    "fit": fit_run,
    "lr": lr_run,
    "dc": lambda: (lambda C: [C[0], C[1], C[n]])(dct2(digit, n)),
    "psnr5": lambda: psnr(digit, idct2_truncated(dct2(digit, n), n, 5)),
    "psnr8": lambda: psnr(digit, idct2_truncated(dct2(digit, n), n, 8)),
})